# Ihminen mukana prosessissa: Ennen-toimintaportit, riskiluokitus ja tarkastuslokit

Tämän oppitunnin README esittelee Ihminen mukana -mallin lyhyellä pätkällä, joka kysyy käyttäjältä hyväksyykö (`APPROVE`) vai hylkääkö (`REJECT`) agentin vastauksen tuottamisen jälkeen. Tämä malli on hyvä lähtökohta, mutta tuotantotason HITL-toteutuksissa tarvitaan yleensä kolme lisäosaa:

1. **Ennen-toimintaportti**, joka suoritetaan **ennen** agentin riskialttiin vaiheen käynnistämistä, jotta kustannukset, peruuttamattomuus ja viive pysyvät hallinnassa.
2. **Riskiluokitus**, jossa matalan riskin toiminnot suoritetaan automaattisesti, keskiriskiset toiminnot hyväksytään erissä ja vain korkean riskin toiminnot edellyttävät ihmisen hyväksyntää.
3. **Tarkastuslokit ja uudelleenkäsittelysilmukka**, joissa jokainen porttipäätös tallennetaan JSONL-muodossa, ja hylkäys johtaa agentin uudelleenpyyntöön rakenteellisen syyn kera sen sijaan, että vain tulostettaisiin `Revising...`.

Tämä muistikirja rakentaa jokaisen näistä samoilla perusrakenteilla kuin `06-system-message-framework.ipynb`. Se toimii kokonaan `DEMO_MODE = True` (ei interaktiivista syötettä) tai oikeilla `input()`-kutsuilla kun `DEMO_MODE = False`. Huom: DEMO_MODE-tilassa kolmannen tavoitteen uudelleenyrittäminen on ohjattu skriptillä, jotta silmukan toiminta näkyy kokonaisuudessaan. Todellinen uudelleentarkistukseen perustuva luokitus vaatii `DEMO_MODE = False` ja käyttäjän.

**Rajauksen ulkopuolella (käsitellään muissa oppitunneissa):** tunnistus ja pääsynhallinta (Oppitunti 06 README uhka 2), työkalukutsun välikerros (Oppitunti 14 MAF syväluotaus), moni-agenttikeskustelumallit.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Kuvio 1: Ennen-toiminto portti

README:n HITL-koodinpätkä kutsuu ensin agenttia, sitten pyytää käyttäjältä hyväksyntää tulokselle. Tämä on **toiminnon jälkeinen** työvirta. Agentti on jo suorittanut, joten LLM-kutsu on jo maksettu, ja kaikki sivuvaikutukset (lähetetty sähköposti, kirjoitettu tietokantarivi, julkaistu kommentti) ovat jo tapahtuneet.

**Ennen-toiminto** -työvirta asettaa portin ennen kuin agentti suorittaa riskialttiin vaiheen. Agentti ehdottaa toimintoa, portti päättää suoritetaanko se, ja vain hyväksynnän jälkeen sivuvaikutus tapahtuu.

| Näkökulma | Toiminnon jälkeinen hyväksyntä (README-koodinpätkä) | Ennen-toiminto portti (tämä muistio) |
|---|---|---|
| Milloin hyväksyntä suoritetaan? | Kun agentti on tuottanut tuloksen | Ennen kuin mikään sivuvaikutus suoritetaan |
| LLM-kustannukset hylkäyksestä | Jo maksettu | Maksetaan vain ehdotuksesta, ei toiminnosta |
| Peruuttamattomat sivuvaikutukset | Mahdollisia (toiminto on jo tapahtunut) | Estetty |
| Tarkastusselkeys | Hyväksyntä on tulostuslause | Hyväksyntä on aikaleimattu JSON-tietue, jossa toiminto ja syy |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Malli 2: Riskiluokittelu

Jokainen toiminto ei vaadi ihmisen hyväksyntää. Julkiseen API:iin kohdistuva lukuoperaatio on erilainen riski kuin asiakkaalle lähetettävä sähköposti. Molempien käsitteleminen samalla tavalla hukkaa operaattorin huomiota ja hidastaa agenttia.

Yksinkertainen 3-tasoinen malli:

| Taso | Esimerkkejä | Hyväksyntäprosessi |
|---|---|---|
| `matala` (vain luku) | Tietokannan haku, lentovaihtoehtojen katselu, julkisen verkkosivun nouto | Automaattinen suoritus, kirjattu tarkastelua varten |
| `keskitaso` (halpa muuttaminen) | Välimuistin tuloksen tallennus, laskurin kasvattaminen, muistutuksen aikataulutus | Automaattinen suoritus, mutta koottu päivittäistä tarkastelua varten |
| `korkea` (ulkopuolelle näkyvä tai peruuttamaton) | Sähköpostin lähetys, kortin veloittaminen, julkiseen kanavaan postaaminen | Estetty ihmisen hyväksyntään asti |

Tämä on yksi luokittelutapa. Tuotantojärjestelmissä käytetään usein tarkempia tasoja (esim. AWS IAM -käyttöoikeustasot, roolipohjaiset pääsytasot). Alla oleva 3-tason versio on pienin hyödyllinen versio agentille, joka yhdistää vain-luku- ja sivuvaikutustoimintoja.

Alla oleva luokittelija käyttää avainsanapohjaista heuristiikkaa, jotta demo pysyy määrällisesti samanlaisena ja halpana. Tuotantojärjestelmässä korvaisit tämän opitulla luokittelijalla tai politiikkamoottorilla.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Malli 3: Tarkastusloki ja tarkistussilmukka

`print("Response approved.")` ei ole tarkastusloki. Luottamuksen takaamiseksi jokainen porttipäätös tulisi tallentaa rakenteellisena tapahtumana, jota voit myöhemmin kysyä, toistaa tai liittää incident-tarkasteluun.

Kaksi osaa:

1. **Lisää pelkästään JSONL-muotoon.** Yksi rivi päätöstä kohden, sisältäen aikaleiman, toimenpiteen, tason, päätöksen ja syyn. Helppo grepata, helppo lähettää myöhemmin oikeaan lokitallennuspaikkaan.
2. **Tarkistussilmukka hylkäyksessä.** Kun portti palauttaa `deny`, agentti kehottaa itseään uudelleen hylkäyssyy kontekstissa, jotta seuraava ehdotus voi välttää ongelman.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Lisäresurssit

Useat muut julkiset projektit toteuttavat näiden HITL-mallien muunnelmia. Vertaa lähestymistapoja löytääksesi omaan pinoosi sopivan:

- **LangChain** human-in-the-loop -työkalukääreet ([dokumentaatio](https://python.langchain.com/docs/integrations/tools/human_tools)): valmiit työkalukääreet, jotka pysäyttävät suorituksen ihmisen syötteelle.
- **AutoGen** `UserProxyAgent` ([v0.2 dokumentaatio](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ rakenteellisesti uudistettu): käyttää agentin roolia ihmisestä edustajana monen agentin keskusteluissa.
- **Semantic Kernel** funktiosuodattimet ([dokumentaatio](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware, joka toimii jokaisen funktion kutsun ympärillä, soveltuu porttien logiikkaan.

Jokainen projekti käsittelee kolmea alatapaa eri tavoin: LangChain käärii ne työkaluiksi, AutoGen käyttää agentin roolia, Semantic Kernel käyttää middleware-suodattimia. Lue yksi tai kaksi toteutusta kokonaisuudessaan ennen oman agenttisi suunnitelman valintaa.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
